# 12 — Pipeline Sonrasi: Zone Secimi + Pipeline + Degerlendirme

Fine-tune BITTI. Bu notebook sadece:
1. Ortami hazirla (Drive + paketler + kod)
2. Zone sec (tarali alan koordinatlari)
3. Pipeline calistir (pretrained + fine-tuned)
4. Ground truth ile P/R/F1
5. Grafik ve tablolar
6. Demo video

**Fine-tune tekrar YAPILMAYACAK** — `best.pt` Drive'da hazir.

---
## ADIM 1: Ortami Hazirla

In [ ]:
# 1.1 Drive bagla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.2 Degiskenler
import os

DRIVE_ROOT = '/content/drive/MyDrive'
VIDEO_DIR = os.path.join(DRIVE_ROOT, 'istanbul_trafik_kayit')
RESULTS_DIR = os.path.join(DRIVE_ROOT, 'tez_sonuclari')
MODELS_DIR = os.path.join(DRIVE_ROOT, 'tez_models')

os.makedirs(RESULTS_DIR, exist_ok=True)

# Fine-tuned model yolu — DOKUNMA, zaten egitildi
FT_BEST = None
for d in sorted(os.listdir(MODELS_DIR), reverse=True):
    candidate = os.path.join(MODELS_DIR, d, 'weights', 'best.pt')
    if os.path.exists(candidate):
        FT_BEST = candidate
        break

if FT_BEST:
    print(f'Fine-tuned model bulundu: {FT_BEST}')
else:
    print('HATA: best.pt bulunamadi! Once 11_master_pipeline ile fine-tune yap.')

# Videolari kontrol et
CAMERAS = ['cam1', 'cam2', 'cam3', 'cam4', 'cam5']
print(f'\nVideolar:')
for cam in CAMERAS:
    found = False
    for ext in ['.mov', '.MOV', '.mp4']:
        if os.path.exists(os.path.join(VIDEO_DIR, cam + ext)):
            print(f'  {cam}{ext} OK')
            found = True
            break
    if not found:
        print(f'  {cam}: BULUNAMADI')

In [ ]:
# 1.3 Pipeline kodunu ac
import zipfile

ZIP_PATH = os.path.join(DRIVE_ROOT, 'pipeline_code.zip')
PROJECT_DIR = '/content/pipeline_code'

if os.path.exists(PROJECT_DIR):
    print(f'Pipeline kodu zaten var: {PROJECT_DIR}')
elif os.path.exists(ZIP_PATH):
    os.makedirs(PROJECT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(PROJECT_DIR)
    print(f'Pipeline kodu cikarildi: {os.listdir(PROJECT_DIR)}')
else:
    print(f'HATA: {ZIP_PATH} bulunamadi! Drive ana klasorune yukle.')

In [ ]:
# 1.4 Paketleri yukle
!pip install ultralytics shapely scikit-learn tqdm seaborn pyyaml -q

import sys
sys.path.insert(0, PROJECT_DIR)

import torch
import cv2
import numpy as np
import json
import time
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from tqdm import tqdm

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Pipeline modullerini import et
from src.tracking.bytetrack_wrapper import ByteTrackWrapper
from src.zones.zone_manager import ZoneManager
from src.violation.trajectory import TrajectoryAnalyzer
from src.violation.severity import SeverityScorer
print('Pipeline modulleri yuklendi OK')

---
## ADIM 2: Zone Secimi (Tarali Alan Koordinatlari)

Her kamera icin tarali alanin kose noktalarini gir.
Asagidaki koordinatlari GUNCELLE.

In [ ]:
# 2.1 Frame'leri goster (koordinat secimi icin)
fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for idx, cam in enumerate(CAMERAS):
    video_path = None
    for ext in ['.mov', '.MOV', '.mp4']:
        candidate = os.path.join(VIDEO_DIR, cam + ext)
        if os.path.exists(candidate):
            video_path = candidate
            break

    if not video_path:
        axes[idx].set_title(f'{cam}: BULUNAMADI')
        continue

    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
    ret, frame = cap.read()
    cap.release()

    if ret:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(frame_rgb)
    axes[idx].set_title(f'{cam}', fontsize=14)
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'camera_frames.png'), dpi=150)
plt.show()

In [ ]:
# 2.2 Interaktif zone secimi
from google.colab import output
from IPython.display import display, HTML
import base64

def interactive_zone_selector(cam_name):
    """Frame goster, tiklanan noktalari topla."""
    video_path = None
    for ext in ['.mov', '.MOV', '.mp4']:
        candidate = os.path.join(VIDEO_DIR, cam_name + ext)
        if os.path.exists(candidate):
            video_path = candidate
            break

    if not video_path:
        print(f'{cam_name}: Video bulunamadi')
        return

    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
    ret, frame = cap.read()
    cap.release()

    h, w = frame.shape[:2]
    scale = min(1.0, 1200 / w)
    if scale < 1.0:
        frame = cv2.resize(frame, (int(w*scale), int(h*scale)))

    _, buffer = cv2.imencode('.jpg', frame)
    img_b64 = base64.b64encode(buffer).decode()

    html = f'''
    <h3>{cam_name} — Tarali alanin koselerine tikla (saat yonunde)</h3>
    <p>En az 4 nokta sec. Bitince asagidaki koordinatlari kopyala.</p>
    <canvas id="canvas_{cam_name}" style="cursor:crosshair"></canvas>
    <p id="points_{cam_name}" style="font-family:monospace; font-size:14px"></p>
    <script>
    (function() {{
        var canvas = document.getElementById('canvas_{cam_name}');
        var ctx = canvas.getContext('2d');
        var img = new Image();
        var points = [];
        var scale = {1.0/scale};
        img.onload = function() {{
            canvas.width = img.width;
            canvas.height = img.height;
            ctx.drawImage(img, 0, 0);
        }};
        img.src = 'data:image/jpeg;base64,{img_b64}';
        canvas.addEventListener('click', function(e) {{
            var rect = canvas.getBoundingClientRect();
            var x = Math.round((e.clientX - rect.left) * scale);
            var y = Math.round((e.clientY - rect.top) * scale);
            points.push([x, y]);
            ctx.drawImage(img, 0, 0);
            for (var i = 0; i < points.length; i++) {{
                var px = points[i][0] / scale;
                var py = points[i][1] / scale;
                ctx.beginPath();
                ctx.arc(px, py, 5, 0, 2*Math.PI);
                ctx.fillStyle = 'red';
                ctx.fill();
                ctx.fillStyle = 'white';
                ctx.font = '14px monospace';
                ctx.fillText('P'+(i+1), px+8, py-5);
            }}
            if (points.length > 1) {{
                ctx.beginPath();
                ctx.moveTo(points[0][0]/scale, points[0][1]/scale);
                for (var i = 1; i < points.length; i++) {{
                    ctx.lineTo(points[i][0]/scale, points[i][1]/scale);
                }}
                ctx.closePath();
                ctx.strokeStyle = 'lime';
                ctx.lineWidth = 2;
                ctx.stroke();
            }}
            var out = "'" + '{cam_name}' + "': " + JSON.stringify(points);
            document.getElementById('points_{cam_name}').innerText = out;
        }});
    }})();
    </script>
    '''
    display(HTML(html))

interactive_zone_selector('cam1')

In [ ]:
# Diger kameralar icin de calistir:
interactive_zone_selector('cam2')

In [ ]:
interactive_zone_selector('cam3')

In [ ]:
interactive_zone_selector('cam4')

In [ ]:
interactive_zone_selector('cam5')

In [ ]:
# 2.3 Koordinatlari buraya yapistir
# interactive_zone_selector ciktisini kopyala-yapistir

ZONE_POLYGONS = {
    'cam1': [[2592,1432],[2758,1538],[1680,1996],[2573,1410],[2586,1442],[2582,1442]],
    'cam2': [[1134,1564],[1183,1565],[1073,2249],[1035,2240],[1022,2237],[999,2374],[1055,2381],[1123,1565],[1037,2170],[1022,2226],[907,3011],[981,3022]],
    'cam3': [[846,887],[702,965],[707,971],[876,915]],
    'cam4': [[1443,2115],[1456,1692],[2243,1328],[2288,1420]],
    'cam5': [[314,1042],[374,1038],[2298,2114],[2739,1925],[592,1218],[854,1234],[464,1138],[963,1438],[963,1438],[1014,1250],[1549,1486],[1520,1717],[2250,1774]],
}

# Zone dosyalarini kaydet
ZONES_DIR = os.path.join(RESULTS_DIR, 'zones')
os.makedirs(ZONES_DIR, exist_ok=True)

for cam, polygon in ZONE_POLYGONS.items():
    zone_data = {
        'camera_id': cam,
        'zones': [{
            'zone_id': 'hatched_area_1',
            'name': f'{cam} tarali alan',
            'polygon': polygon,
            'type': 'hatched_area',
        }]
    }
    path = os.path.join(ZONES_DIR, f'{cam}.json')
    with open(path, 'w') as f:
        json.dump(zone_data, f, indent=2)
    print(f'{cam} zone kaydedildi: {path}')

---
## ADIM 3: Pipeline Calistir (5 Video x 2 Model)

In [ ]:
# 3.1 Pipeline fonksiyonu

def run_pipeline_on_video(video_path, zone_file, model_path='yolov8s.pt', label='pretrained'):
    """Tek bir video uzerinde pipeline calistir."""
    cam_name = os.path.splitext(os.path.basename(video_path))[0]
    print(f'\n{"="*60}')
    print(f'Pipeline: {cam_name} ({label})')
    print(f'{"="*60}')

    tracker = ByteTrackWrapper(
        model_path=model_path,
        conf=0.35, iou=0.45,
        classes=[2, 3, 5, 7],
        img_size=640, half=False,
    )
    zone_mgr = ZoneManager(zone_file=zone_file)
    trajectory = TrajectoryAnalyzer()
    severity = SeverityScorer()

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f'  FPS: {fps:.0f} | Sure: {total/fps:.0f}s | Cozunurluk: {w}x{h}')

    violations = []
    frame_num = 0
    start_time = time.time()
    recorded_ids = set()

    pbar = tqdm(total=total, desc=f'{cam_name} ({label})')

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        tracked = tracker.update(None, frame)

        for track in tracked:
            track_id = track.track_id
            bbox = track.bbox
            class_name = track.detection.class_name

            cx = (bbox[0] + bbox[2]) / 2
            cy = bbox[3]

            in_zone, zone_id = zone_mgr.is_point_in_zone((cx, cy))
            trajectory.update(track_id, (cx, cy), in_zone)

            if in_zone:
                zone_polygon = None
                for z in zone_mgr.zones:
                    if z.zone_id == zone_id:
                        zone_polygon = z.polygon
                        break
                if zone_polygon is None and zone_mgr.zones:
                    zone_polygon = zone_mgr.zones[0].polygon

                metrics = trajectory.compute_metrics(track_id, zone_polygon)

                if metrics.in_zone_frames >= 5 and track_id not in recorded_ids:
                    sev_result = severity.score(metrics)
                    recorded_ids.add(track_id)
                    violations.append({
                        'track_id': track_id,
                        'frame': frame_num,
                        'timestamp': round(frame_num / fps, 1),
                        'vehicle_class': class_name,
                        'bbox': bbox.tolist() if hasattr(bbox, 'tolist') else list(bbox),
                        'severity_score': sev_result.score,
                        'severity_level': sev_result.level.value,
                        'violation_type': sev_result.violation_type.value,
                        'zone_id': zone_id,
                    })

        frame_num += 1
        pbar.update(1)

    pbar.close()
    cap.release()

    elapsed = time.time() - start_time
    proc_fps = frame_num / elapsed if elapsed > 0 else 0

    result = {
        'camera': cam_name,
        'model': label,
        'total_frames': frame_num,
        'duration_sec': round(frame_num / fps, 1),
        'processing_fps': round(proc_fps, 1),
        'processing_time_sec': round(elapsed, 1),
        'total_violations': len(violations),
        'violations': violations,
    }

    out_path = os.path.join(RESULTS_DIR, f'{cam_name}_{label}_results.json')
    with open(out_path, 'w') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f'  Toplam ihlal: {len(violations)}')
    print(f'  Isleme hizi: {proc_fps:.1f} FPS ({elapsed:.0f}s)')

    return result

In [ ]:
# 3.2 PRETRAINED model ile calistir
pretrained_results = {}

for cam in CAMERAS:
    video_path = None
    for ext in ['.mov', '.MOV', '.mp4']:
        candidate = os.path.join(VIDEO_DIR, cam + ext)
        if os.path.exists(candidate):
            video_path = candidate
            break
    if not video_path:
        print(f'{cam}: Video bulunamadi, atlaniyor.')
        continue

    zone_file = os.path.join(ZONES_DIR, f'{cam}.json')
    result = run_pipeline_on_video(video_path, zone_file, model_path='yolov8s.pt', label='pretrained')
    pretrained_results[cam] = result

print(f'\nPRETRAINED TAMAMLANDI: {len(pretrained_results)} video')

In [ ]:
# 3.3 FINE-TUNED model ile calistir
finetuned_results = {}

for cam in CAMERAS:
    video_path = None
    for ext in ['.mov', '.MOV', '.mp4']:
        candidate = os.path.join(VIDEO_DIR, cam + ext)
        if os.path.exists(candidate):
            video_path = candidate
            break
    if not video_path:
        continue

    zone_file = os.path.join(ZONES_DIR, f'{cam}.json')
    result = run_pipeline_on_video(video_path, zone_file, model_path=FT_BEST, label='finetuned')
    finetuned_results[cam] = result

print(f'\nFINE-TUNED TAMAMLANDI: {len(finetuned_results)} video')

---
## ADIM 4: Ground Truth

Videolari izle, ihlalleri yaz. P/R/F1 icin ZORUNLU.

In [ ]:
# 4.1 Ground truth — videolari izleyip doldur
# Her ihlal icin: baslangic saniye, bitis saniye, arac tipi, ihlal tipi

GROUND_TRUTH = {
    'cam1': [
        # {'start_sec': 15.0, 'end_sec': 18.5, 'vehicle_class': 'car', 'type': 'KAYNAK'},
        # Videoyu izleyip buraya yaz
    ],
    'cam2': [],
    'cam3': [],
    'cam4': [],
    'cam5': [],
}

# Kaydet
gt_dir = os.path.join(RESULTS_DIR, 'ground_truth')
os.makedirs(gt_dir, exist_ok=True)

for cam, gt_list in GROUND_TRUTH.items():
    gt_data = {'video': cam, 'annotator': 'Sertac Akalin', 'violations': gt_list}
    with open(os.path.join(gt_dir, f'{cam}_gt.json'), 'w') as f:
        json.dump(gt_data, f, indent=2, ensure_ascii=False)

total_gt = sum(len(v) for v in GROUND_TRUTH.values())
print(f'Ground truth: {total_gt} ihlal kaydi')
if total_gt == 0:
    print('UYARI: Ground truth BOS! Videolari izleyip ihlalleri yukardaki sozluge yaz.')

In [ ]:
# 4.2 P/R/F1 hesaplama

def evaluate_camera(pipeline_result, ground_truth, tolerance_sec=3.0):
    detected = pipeline_result.get('violations', [])
    gt = ground_truth
    if not gt:
        return {'precision': 0, 'recall': 0, 'f1': 0, 'tp': 0, 'fp': len(detected), 'fn': 0}

    gt_matched = [False] * len(gt)
    det_matched = [False] * len(detected)

    for i, det in enumerate(detected):
        det_time = det['timestamp']
        for j, g in enumerate(gt):
            if gt_matched[j]:
                continue
            if g['start_sec'] - tolerance_sec <= det_time <= g['end_sec'] + tolerance_sec:
                gt_matched[j] = True
                det_matched[i] = True
                break

    tp = sum(det_matched)
    fp = len(detected) - tp
    fn = len(gt) - sum(gt_matched)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {'precision': round(precision, 4), 'recall': round(recall, 4), 'f1': round(f1, 4),
            'tp': tp, 'fp': fp, 'fn': fn,
            'total_detected': len(detected), 'total_gt': len(gt)}

eval_results_pre = {}
eval_results_ft = {}

for cam in CAMERAS:
    gt = GROUND_TRUTH.get(cam, [])
    if not gt:
        continue
    if cam in pretrained_results:
        eval_results_pre[cam] = evaluate_camera(pretrained_results[cam], gt)
    if cam in finetuned_results:
        eval_results_ft[cam] = evaluate_camera(finetuned_results[cam], gt)

if eval_results_ft:
    print('=== FINE-TUNED SONUCLARI ===')
    for cam, ev in eval_results_ft.items():
        print(f'  {cam}: P={ev["precision"]:.2f} R={ev["recall"]:.2f} F1={ev["f1"]:.2f} (TP={ev["tp"]} FP={ev["fp"]} FN={ev["fn"]})')
else:
    print('Ground truth bos — P/R/F1 hesaplanamadi. Adim 4.1 i doldur.')

---
## ADIM 5: Grafik ve Tablolar

In [ ]:
# 5.1 Ihlal ozet tablosu
sns.set_style('whitegrid')

rows = []
for cam in CAMERAS:
    for label, results in [('Pretrained', pretrained_results), ('Fine-tuned', finetuned_results)]:
        if cam in results:
            r = results[cam]
            rows.append({
                'Kamera': cam,
                'Model': label,
                'Ihlal Sayisi': r['total_violations'],
                'FPS': r['processing_fps'],
                'Sure (s)': r['duration_sec'],
            })

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))
df_summary.to_csv(os.path.join(RESULTS_DIR, 'ihlal_ozet.csv'), index=False)

In [ ]:
# 5.2 Siddet skoru dagilimi
all_scores = []
all_types = []
all_levels = []

for cam, result in finetuned_results.items():
    for v in result.get('violations', []):
        all_scores.append(v['severity_score'])
        all_types.append(v['violation_type'])
        all_levels.append(v['severity_level'])

if all_scores:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Histogram
    axes[0].hist(all_scores, bins=20, color='steelblue', edgecolor='white')
    axes[0].set_xlabel('Siddet Skoru')
    axes[0].set_ylabel('Sayi')
    axes[0].set_title('Siddet Skoru Dagilimi')
    axes[0].axvline(x=25, color='green', linestyle='--', alpha=0.7, label='DUSUK/ORTA')
    axes[0].axvline(x=50, color='orange', linestyle='--', alpha=0.7, label='ORTA/YUKSEK')
    axes[0].axvline(x=75, color='red', linestyle='--', alpha=0.7, label='YUKSEK/KRITIK')
    axes[0].legend()

    # Ihlal tipi pasta
    type_counts = pd.Series(all_types).value_counts()
    axes[1].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%', startangle=90)
    axes[1].set_title('Ihlal Tipi Dagilimi')

    # Seviye bar
    level_order = ['DÜŞÜK', 'ORTA', 'YÜKSEK', 'KRİTİK']
    level_colors = ['green', 'orange', 'red', 'darkred']
    level_counts = pd.Series(all_levels).value_counts()
    ordered = [level_counts.get(l, 0) for l in level_order]
    axes[2].bar(level_order, ordered, color=level_colors)
    axes[2].set_title('Siddet Seviyesi Dagilimi')
    axes[2].set_ylabel('Sayi')

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'grafik_siddet_dagilimi.png'), dpi=150)
    plt.show()
else:
    print('Ihlal bulunamadi — grafik uretilemedi.')

In [ ]:
# 5.3 Kamera bazli ihlal karsilastirmasi
if pretrained_results and finetuned_results:
    fig, ax = plt.subplots(figsize=(10, 6))
    x = range(len(CAMERAS))
    w = 0.35
    pre_counts = [pretrained_results.get(c, {}).get('total_violations', 0) for c in CAMERAS]
    ft_counts = [finetuned_results.get(c, {}).get('total_violations', 0) for c in CAMERAS]
    ax.bar([i - w/2 for i in x], pre_counts, w, label='Pretrained', color='gray')
    ax.bar([i + w/2 for i in x], ft_counts, w, label='Fine-tuned', color='steelblue')
    ax.set_xlabel('Kamera')
    ax.set_ylabel('Ihlal Sayisi')
    ax.set_title('Pretrained vs Fine-tuned: Kamera Bazli Ihlal Sayisi')
    ax.set_xticks(x)
    ax.set_xticklabels(CAMERAS)
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'grafik_model_karsilastirma.png'), dpi=150)
    plt.show()

---
## ADIM 6: Demo Video

In [ ]:
# 6.1 Demo video uret
DEMO_CAM = 'cam1'
DEMO_MODEL = FT_BEST

def create_demo_video(cam_name, model_path, zone_file, max_frames=None):
    video_path = None
    for ext in ['.mov', '.MOV', '.mp4']:
        candidate = os.path.join(VIDEO_DIR, cam_name + ext)
        if os.path.exists(candidate):
            video_path = candidate
            break

    tracker = ByteTrackWrapper(
        model_path=model_path,
        conf=0.35, iou=0.45,
        classes=[2, 3, 5, 7],
        img_size=640, half=False,
    )
    zone_mgr = ZoneManager(zone_file=zone_file)
    trajectory = TrajectoryAnalyzer()
    severity_scorer = SeverityScorer()

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total = min(total, max_frames)

    out_path = os.path.join(RESULTS_DIR, f'demo_{cam_name}.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, fps, (w, h))

    zone_polys = zone_mgr.get_zone_polygons_for_drawing()
    active_violations = {}
    frame_num = 0

    for _ in tqdm(range(total), desc=f'Demo {cam_name}'):
        ret, frame = cap.read()
        if not ret:
            break

        overlay = frame.copy()
        for name, poly in zone_polys:
            pts = poly.reshape((-1, 1, 2)).astype(np.int32)
            cv2.fillPoly(overlay, [pts], (0, 165, 255))
            cv2.polylines(frame, [pts], True, (0, 165, 255), 2)
        frame = cv2.addWeighted(overlay, 0.2, frame, 0.8, 0)

        tracked = tracker.update(None, frame)

        for track in tracked:
            tid = track.track_id
            bbox = track.bbox
            x1, y1, x2, y2 = [int(v) for v in bbox]
            cx, cy = (x1+x2)//2, y2

            in_zone, zone_id = zone_mgr.is_point_in_zone((cx, cy))
            trajectory.update(tid, (cx, cy), in_zone)

            if in_zone:
                zone_polygon = None
                for z in zone_mgr.zones:
                    if z.zone_id == zone_id:
                        zone_polygon = z.polygon
                        break
                if zone_polygon is None and zone_mgr.zones:
                    zone_polygon = zone_mgr.zones[0].polygon

                metrics = trajectory.compute_metrics(tid, zone_polygon)
                if metrics.in_zone_frames >= 5:
                    sev_result = severity_scorer.score(metrics)
                    active_violations[tid] = {
                        'score': sev_result.score,
                        'level': sev_result.level.value,
                        'type': sev_result.violation_type.value,
                        'class': track.detection.class_name,
                    }

            if tid in active_violations:
                v = active_violations[tid]
                color = (0, 0, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
                label = f\"IHLAL {v['class']} S:{v['score']:.0f}\"
                cv2.putText(frame, label, (x1, y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            else:
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        info = f'Frame: {frame_num} | Ihlal: {len(active_violations)}'
        cv2.rectangle(frame, (0, 0), (400, 35), (0, 0, 0), -1)
        cv2.putText(frame, info, (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        out.write(frame)
        frame_num += 1

    cap.release()
    out.release()
    print(f'Demo video: {out_path}')
    return out_path

zone_file = os.path.join(ZONES_DIR, f'{DEMO_CAM}.json')
demo_path = create_demo_video(DEMO_CAM, DEMO_MODEL, zone_file)

---
## ADIM 7: Sonuclari Kontrol Et

In [ ]:
# 7.1 Uretilen dosyalari listele
print(f'\n{"="*60}')
print(f'SONUCLAR: {RESULTS_DIR}')
print(f'{"="*60}')

for root, dirs, files in os.walk(RESULTS_DIR):
    level = root.replace(RESULTS_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for file in sorted(files):
        fpath = os.path.join(root, file)
        size = os.path.getsize(fpath)
        if size > 1024*1024:
            size_str = f'{size/(1024*1024):.1f} MB'
        elif size > 1024:
            size_str = f'{size/1024:.1f} KB'
        else:
            size_str = f'{size} B'
        print(f'{subindent}{file} ({size_str})')

In [ ]:
# 7.2 Ozet
print('\n=== PROJE OZETI ===')
print(f'Fine-tuned model: {FT_BEST}')
print(f'Islenen video: {len(finetuned_results)}')

total_violations = sum(r['total_violations'] for r in finetuned_results.values())
print(f'Toplam ihlal (fine-tuned): {total_violations}')

if eval_results_ft:
    avg_f1 = sum(e['f1'] for e in eval_results_ft.values()) / len(eval_results_ft)
    print(f'Ortalama F1 (fine-tuned): {avg_f1:.2f}')

print(f'\nSonuclar: {RESULTS_DIR}')
print('Bitti!')